# Making SIMSOPT GPU native: feasibility continuation

Select **Runtime > Change runtime type > GPU**, then run all cells. This workflow first sweeps SciPy L-BFGS-B memory and line-search settings, then runs synchronized engineering-penalty continuation from a common CPU/GPU state. It records absolute engineering-feasibility gates, complete final CPU/GPU metrics, and final VTS/VTU files for ParaView. A failed feasibility or stationarity gate is a scientific result and does not prevent artifact download.

In [ ]:
import subprocess

subprocess.run(["nvidia-smi"], check=True)

In [ ]:
import importlib
import os
import sys
from pathlib import Path

repo = Path("/content/simsopt")
if not repo.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", "gpu-native-objective", "https://github.com/PedroFranciscoGil/simsopt.git", str(repo)], check=True)
else:
    subprocess.run(["git", "fetch", "origin", "gpu-native-objective"], cwd=repo, check=True)
    subprocess.run(["git", "switch", "gpu-native-objective"], cwd=repo, check=True)
    subprocess.run(["git", "pull", "--ff-only"], cwd=repo, check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".", "pytest", "pyevtk"], cwd=repo, check=True)
os.chdir(repo)
source_root = repo / "src"
sys.path.insert(0, str(source_root))
for module_name in tuple(sys.modules):
    if module_name == "simsopt" or module_name.startswith("simsopt."):
        del sys.modules[module_name]
importlib.invalidate_caches()
revision = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=repo, text=True).strip()
print(revision)

In [ ]:
import jax
import simsopt
from simsopt.gpu import backend_report

resolved_package = Path(simsopt.__file__).resolve()
print(f"Imported SIMSOPT from {resolved_package}")
assert source_root in resolved_package.parents, resolved_package
report = backend_report()
print(report)
assert jax.default_backend() == "gpu", report

In [ ]:
subprocess.run([sys.executable, "-m", "pytest", "-q", "tests/gpu"], check=True)

In [ ]:
import shutil

artifact_root = Path("/content/simsopt-feasibility-study")
if artifact_root.exists():
    shutil.rmtree(artifact_root)
artifact_root.mkdir()
sweep_root = artifact_root / "solver-sweep"
env = os.environ.copy()
env["OMP_NUM_THREADS"] = "1"
env["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
subprocess.run([sys.executable, "benchmarks/gpu/sweep_solver_feasibility.py", "--problem", "stress", "--maxiter", "25", "--maxcor-values", "10,100,300", "--maxls-values", "20,50", "--current-scale", "100000", "--target-tile-size", "1024", "--source-tile-size", "4320", "--output-dir", str(sweep_root)], cwd=repo, env=env, check=True)

In [ ]:
import json

sweep = json.loads((sweep_root / "solver-feasibility-sweep.json").read_text())
assert sweep["schema_version"] == 1
assert len(sweep["candidates"]) == 6
winner = sweep["winner"]
print(json.dumps({"ranking": sweep["ranking"], "winner": winner}, indent=2))

In [ ]:
continuation_root = artifact_root / "penalty-continuation"
subprocess.run([sys.executable, "benchmarks/gpu/run_penalty_continuation.py", "--problem", "stress", "--penalty-multipliers", "1,10,100", "--stage-maxiters", "100,100,200", "--maxcor", str(winner["maxcor"]), "--maxls", str(winner["maxls"]), "--current-scale", "100000", "--target-tile-size", "1024", "--source-tile-size", "4320", "--output-dir", str(continuation_root)], cwd=repo, env=env, check=True)

In [ ]:
continuation = json.loads((continuation_root / "penalty-continuation-summary.json").read_text())
assert continuation["schema_version"] == 1
assert continuation["acceptance"]["all_stage_gpu_backend"]
final_result = json.loads((continuation_root / continuation["final_result_file"]).read_text())
assert final_result["schema_version"] == 4
assert "final_metrics" in final_result["cpu"]
assert "final_metrics_from_cpu_oracle" in final_result["gpu"]
expected_files = ["cpu_final_surface.vts", "cpu_final_coils.vtu", "gpu_final_surface.vts", "gpu_final_coils.vtu"]
for filename in expected_files:
    path = continuation_root / filename
    assert path.is_file() and path.stat().st_size > 0, path
decision = {"accepted": continuation["accepted"], "acceptance": continuation["acceptance"], "totals": continuation["totals"], "cpu_final_metrics": final_result["cpu"]["final_metrics"], "gpu_final_metrics": final_result["gpu"]["final_metrics_from_cpu_oracle"]}
print(json.dumps(decision, indent=2))

## Interpretation and ParaView

The continuation is accepted only if every listed gate passes. If feasibility or stationarity is false, send the archive back unchanged; the failed run still identifies the next conditioning or constraint-handling decision. For visual inspection, open a final `*_surface.vts`, choose **Surface**, and color by `B_dot_n_over_abs_B` or `abs_B_dot_n_over_abs_B`; add the matching `*_coils.vtu`.

In [ ]:
from google.colab import files

archive = shutil.make_archive("/content/simsopt-feasibility-study", "zip", artifact_root)
files.download(archive)